# Common Pitfalls — Exercises

Companion to the note [Common Pitfalls](Common%20Pitfalls.md).

Diagnose-the-bug practice for every row of the pitfalls table: each exercise shows code that *demonstrates* a pitfall (leakage through preprocessing or the target, shuffled time series, tuning on the test set, accuracy on imbalanced data, unscaled distances, groups split across folds, missing baselines, shape bugs, train/eval mode, numerical overflow) and asks you to measure the damage and write the fix.

**18 exercises** · 🧠 Concept ×4 · 💻 Code ×11 · ✍️ By hand ×3

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import (train_test_split, cross_val_score, KFold, StratifiedKFold,
                                     TimeSeriesSplit, GroupKFold)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score, precision_score,
                             recall_score, f1_score, mean_squared_error, mean_absolute_error, balanced_accuracy_score)
from scipy.special import logsumexp as sp_logsumexp, log_expit
from scipy.stats import norm
rng = np.random.default_rng(0)

## Part A · Spotting pitfalls

### Exercise 1 · Spot the pitfalls
*🧠 Concept · ★☆☆*

A classmate reports: *"I standardised the whole fraud dataset (0.5% fraud), selected the 20 features most correlated with the label,
then did a random 80/20 split. I tried 40 hyperparameter settings and report the best test accuracy: 99.6%. Transactions of the same
customer appear in both splits. This proves our features cause fraud."*

List every pitfall from the note's table that appears, and the fix for each.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

There are at least six. Go row by row through the table in [[Common Pitfalls]].

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Leakage via preprocessing**: scaling and especially *feature selection with the label* on all data before splitting. Fix: split first, put preprocessing in a `Pipeline`.
2. **Tuning on the test set**: picking the best of 40 settings by test accuracy. Fix: validation set / (nested) CV, touch the test set once.
3. **Accuracy on imbalanced data**: always predicting "not fraud" already gives 99.5%. Fix: PR-AUC, F1, recall at fixed precision, and compare to the dummy baseline.
4. **Ignoring the baseline**: 99.6% vs 99.5% is almost nothing.
5. **Groups across train and test**: the same customer on both sides. Fix: `GroupKFold` / group split by customer.
6. **Random split for transactional (time-ordered) data**: future information leaks. Fix: time-aware split.
7. **Correlation ≠ causation**: predictive features are not causes ([[Causal Inference]]).

</details>

## Part B · Leakage

### Exercise 2 · Feature selection before CV
*💻 Code · ★☆☆*

The data below is **pure noise**: 100 samples, 2000 random features, random labels. The starter shows the leaky protocol: select the 20 "best"
features on all data, then cross-validate. Store its score in `leaky_cv` (given), then write the **correct** protocol with a pipeline
so that selection happens inside each fold, and store its score in `pipe_cv`.

In [ ]:
X_noise = rng.normal(size=(100, 2000))
y_noise = rng.integers(0, 2, 100)

# --- the bug: selection sees the labels of the validation folds
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky_cv = cross_val_score(LogisticRegression(), X_sel, y_noise, cv=5).mean()
print('leaky CV accuracy:', leaky_cv)

pipe_cv = None   # TODO: correct protocol

check('leaky protocol looks good on pure noise', leaky_cv > 0.7, True)
if pipe_cv is not None:
    check('pipeline gives chance level', abs(pipe_cv - 0.5) < 0.15, True)
else:
    check('pipeline gives chance level', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())` and pass the **raw** `X_noise` to `cross_val_score`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With 2000 noise features, some are correlated with the 100 labels by chance; choosing them using **all** labels leaks the validation labels,
so CV reports ~0.78 on data with no signal. In the pipeline the selector is re-fit on the training folds only, and the score falls to chance (~0.5–0.6).
Plain standardisation before splitting leaks much less (only means/variances), but the pipeline habit protects you from both.

```python
X_noise = rng.normal(size=(100, 2000))
y_noise = rng.integers(0, 2, 100)
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky_cv = cross_val_score(LogisticRegression(), X_sel, y_noise, cv=5).mean()
pipe_cv = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression()), X_noise, y_noise, cv=5).mean()
print(leaky_cv, pipe_cv)
```

</details>

### Exercise 3 · Hunting a target leak
*💻 Code · ★☆☆*

Someone added a column to the breast-cancer data that was computed *after* diagnosis (it is basically the label plus noise) and hid it among
the others. Your model gets suspiciously close to 100% accuracy. Find the leaking column by computing the ROC-AUC of **each single feature**
(use $\max(\mathrm{AUC}, 1-\mathrm{AUC})$) and store its index in `leak_col`.

In [ ]:
_Xbc, y_bc = load_breast_cancer(return_X_y=True)
_leak = y_bc + 0.3 * rng.normal(size=len(y_bc))
_pos = rng.integers(0, _Xbc.shape[1] + 1)
X_bc = np.insert(_Xbc, _pos, _leak, axis=1)   # one column is now a leak

leak_col = None

check('leaking column found', leak_col, _pos)

<details>
<summary><b>💡 Hint</b></summary>

`aucs = [roc_auc_score(y_bc, X_bc[:, j]) for j in range(X_bc.shape[1])]`, fold with `max(a, 1-a)` and take the argmax.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The leak has a single-feature AUC near 0.99, clearly above the best genuine feature (~0.97). In practice: look at single-feature AUCs and
feature importances, ask *when* each column becomes available, and be suspicious of any result that is too good to be true.

```python
_Xbc, y_bc = load_breast_cancer(return_X_y=True)
_leak = y_bc + 0.3 * rng.normal(size=len(y_bc))
_pos = rng.integers(0, _Xbc.shape[1] + 1)
X_bc = np.insert(_Xbc, _pos, _leak, axis=1)
aucs = np.array([roc_auc_score(y_bc, X_bc[:, j]) for j in range(X_bc.shape[1])])
aucs = np.maximum(aucs, 1 - aucs)
leak_col = int(np.argmax(aucs))
print(leak_col, np.sort(aucs)[-3:])
```

</details>

### Exercise 4 · Shuffled CV on a time series
*💻 Code · ★★☆*

A random walk $s_t$ is modelled with $k$-NN regression on the time index $t$. Compare the mean $R^2$ of
(a) `KFold(5, shuffle=True, random_state=0)` and (b) `TimeSeriesSplit(5)`. Store them as `r2_shuffled` and `r2_time`. Which number would you
report for a model that has to forecast the future, and why is the other one so optimistic?

In [ ]:
t_idx = np.arange(500.)[:, None]
series = np.cumsum(rng.normal(size=500))
r2_shuffled = None
r2_time = None

if r2_shuffled is not None:
    check('shuffled CV looks excellent', r2_shuffled > 0.9, True)
    check('forward-chaining CV is honest (poor)', r2_time < 0.5, True)
else:
    check('shuffled CV looks excellent', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`cross_val_score(KNeighborsRegressor(5), t_idx, series, cv=...)` with default scoring $R^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Shuffled CV puts time steps $t-1$ and $t+1$ in training when predicting $t$: the model only **interpolates** a smooth curve, so $R^2\approx0.97$.
`TimeSeriesSplit` always trains on the past and tests on the future, which is the real task: $R^2$ is strongly negative (a $k$-NN on $t$ can only
repeat the last values it saw). Report the time-aware number, and compare it to a naive "last value" baseline ([[Cross-Validation and Model Selection]]).

```python
t_idx = np.arange(500.)[:, None]
series = np.cumsum(rng.normal(size=500))
r2_shuffled = cross_val_score(KNeighborsRegressor(5), t_idx, series, cv=KFold(5, shuffle=True, random_state=0)).mean()
r2_time = cross_val_score(KNeighborsRegressor(5), t_idx, series, cv=TimeSeriesSplit(5)).mean()
print(r2_shuffled, r2_time)
```

</details>

### Exercise 5 · Duplicates and groups across folds
*💻 Code · ★★☆*

50 patients contribute 10 scans each. Every patient has a personal "fingerprint" in the features, and the label is per patient (random here,
so there is **no** generalisable signal). Compare 1-NN accuracy under `KFold(5, shuffle=True, random_state=0)` (`acc_kfold`) and
`GroupKFold(5)` with the patient id as group (`acc_group`).

In [ ]:
fingerprint = rng.normal(size=(50, 20))
patient_label = rng.integers(0, 2, 50)
groups = np.repeat(np.arange(50), 10)
X_pat = fingerprint[groups] + 0.3 * rng.normal(size=(500, 20))
y_pat = patient_label[groups]
acc_kfold = None
acc_group = None

if acc_kfold is not None:
    check('KFold memorises patients', acc_kfold > 0.9, True)
    check('GroupKFold reveals chance level', acc_group < 0.65, True)
else:
    check('KFold memorises patients', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`cross_val_score(KNeighborsClassifier(1), X_pat, y_pat, cv=GroupKFold(5), groups=groups)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With random folds, other scans of the same patient are in training, so 1-NN finds the patient and copies the label: ~100%. With `GroupKFold`
each patient is entirely in one fold, and accuracy collapses to ~0.5, the truth. Same story for near-duplicate images, several sessions per user,
or augmented copies of one sample.

```python
fingerprint = rng.normal(size=(50, 20))
patient_label = rng.integers(0, 2, 50)
groups = np.repeat(np.arange(50), 10)
X_pat = fingerprint[groups] + 0.3 * rng.normal(size=(500, 20))
y_pat = patient_label[groups]
acc_kfold = cross_val_score(KNeighborsClassifier(1), X_pat, y_pat, cv=KFold(5, shuffle=True, random_state=0)).mean()
acc_group = cross_val_score(KNeighborsClassifier(1), X_pat, y_pat, cv=GroupKFold(5), groups=groups).mean()
print(acc_kfold, acc_group)
```

</details>

## Part C · Evaluation and model selection

### Exercise 6 · How lucky can the test set make you?
*✍️ By hand · ★★☆*

A useless classifier (true accuracy 0.5) is evaluated on a test set of $n=100$. Using the normal approximation of the binomial,
(1) what is the standard deviation of its test accuracy, and (2) what is the probability that **at least one** of 20 such independent
useless models scores **above 0.6**? (Treat "above 0.6" as $Z>2$.)

In [ ]:
sd_acc = None
p_any_above = None

check('sd of accuracy', sd_acc, np.sqrt(0.25 / 100))
check('P(best of 20 > 0.6)', p_any_above, 1 - norm.cdf(2) ** 20, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\mathrm{sd}=\sqrt{p(1-p)/n}$. $P(Z>2)\approx0.0228$; $P(\text{at least one})=1-(1-0.0228)^{20}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathrm{sd}=\sqrt{0.25/100}=0.05$, so $0.6$ is two standard deviations above chance. $P(\text{one model}>0.6)\approx0.0228$, and
$1-0.9772^{20}\approx0.369$. More than a third of the time, picking the best of 20 junk models on the test set yields an apparently real result.
(The exact binomial $P(X>60)=0.0176$ gives $\approx0.30$; same message.)

```python
sd_acc = 0.05
p_any_above = 1 - (1 - 0.02275) ** 20
```

</details>

### Exercise 7 · Tuning on the test set
*💻 Code · ★★☆*

Simulate the previous exercise: noise data, 200 training and 100 test samples. Try 50 "configurations" (a 1-nearest-neighbour classifier on a random subset of
5 of the 30 features), pick the one with the best **test** accuracy (`best_test_acc`), then evaluate that same model on 10,000 fresh samples
(`fresh_acc`). How should the selection have been done?

In [ ]:
X_tr, y_tr = rng.normal(size=(200, 30)), rng.integers(0, 2, 200)
X_te, y_te = rng.normal(size=(100, 30)), rng.integers(0, 2, 100)
X_fresh, y_fresh = rng.normal(size=(10000, 30)), rng.integers(0, 2, 10000)
best_test_acc = None
fresh_acc = None

if best_test_acc is not None:
    check('best-of-50 test accuracy is inflated', best_test_acc > 0.56, True)
    check('fresh data reveals chance level', fresh_acc, 0.5, tol=0.03)
else:
    check('best-of-50 test accuracy is inflated', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Loop over 50 configs: `f = rng.choice(30, 5, replace=False)`; keep the model and `f` with the highest `score(X_te[:, f], y_te)`; `KNeighborsClassifier(1).fit(X_tr[:, f], y_tr)` is one configuration.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The maximum of 50 noisy test scores is around 0.6, while the selected model's real accuracy is 0.5. Once the test set is used for **selection**
its score is no longer an unbiased estimate. Correct protocol: choose among configurations with a validation split or (nested) cross-validation on the
training data, then evaluate the single chosen model once on the test set.

```python
X_tr, y_tr = rng.normal(size=(200, 30)), rng.integers(0, 2, 200)
X_te, y_te = rng.normal(size=(100, 30)), rng.integers(0, 2, 100)
X_fresh, y_fresh = rng.normal(size=(10000, 30)), rng.integers(0, 2, 10000)
best = (-1, None, None)
for _ in range(50):
    f = rng.choice(30, 5, replace=False)
    m = KNeighborsClassifier(1).fit(X_tr[:, f], y_tr)
    a = m.score(X_te[:, f], y_te)
    if a > best[0]:
        best = (a, m, f)
best_test_acc = best[0]
fresh_acc = best[1].score(X_fresh[:, best[2]], y_fresh)
print(best_test_acc, fresh_acc)
```

</details>

### Exercise 8 · The accuracy paradox
*✍️ By hand · ★☆☆*

1000 transactions, 20 frauds. Classifier A always says "not fraud". Classifier B has TP $=12$, FP $=30$, FN $=8$, TN $=950$.
Compute the accuracy of A and B, and precision, recall and $F_1$ of B. Which classifier is more useful?

In [ ]:
acc_A = None
acc_B = None
prec_B = None
rec_B = None
f1_B = None

_y = np.r_[np.ones(20), np.zeros(980)]
_pB = np.r_[np.ones(12), np.zeros(8), np.ones(30), np.zeros(950)]
check('accuracy A', acc_A, accuracy_score(_y, np.zeros(1000)))
check('accuracy B', acc_B, accuracy_score(_y, _pB))
check('precision B', prec_B, precision_score(_y, _pB), tol=1e-3)
check('recall B', rec_B, recall_score(_y, _pB))
check('F1 B', f1_B, f1_score(_y, _pB), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Accuracy $=(TP+TN)/n$, precision $=TP/(TP+FP)$, recall $=TP/(TP+FN)$, $F_1=2PR/(P+R)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

A: accuracy $980/1000=0.98$, recall 0. B: accuracy $962/1000=0.962$ (**lower** than A!), precision $12/42=0.286$, recall $12/20=0.6$,
$F_1=2\cdot0.286\cdot0.6/0.886=0.387$. B catches 60% of the fraud and is far more useful; accuracy rewards the majority class
([[Model Evaluation and Metrics]]).

```python
acc_A = 0.98
acc_B = 0.962
prec_B = 12/42
rec_B = 0.6
f1_B = 2 * (12/42) * 0.6 / (12/42 + 0.6)
```

</details>

### Exercise 9 · PR-AUC from scratch
*💻 Code · ★★☆*

Implement `average_precision(y, scores)` $=\sum_n (R_n-R_{n-1})P_n$, where $(P_n, R_n)$ are precision and recall when predicting
positive for the top-$n$ scores. On the imbalanced data below (≈2% positives), compare ROC-AUC and AP of a logistic regression, and the AP of
a random scorer. Why is ROC-AUC flattering here?

In [ ]:
X_imb, y_imb = make_classification(n_samples=5000, n_features=10, weights=[0.98], flip_y=0.01, random_state=1)
Xa, Xb_, ya, yb = train_test_split(X_imb, y_imb, test_size=0.5, stratify=y_imb, random_state=0)
scores_imb = LogisticRegression(max_iter=1000).fit(Xa, ya).decision_function(Xb_)

def average_precision(y, scores):
    return None

ap_model = average_precision(yb, scores_imb)
ap_random = None   # AP of rng.uniform scores

check('AP matches sklearn', ap_model, average_precision_score(yb, scores_imb))
if ap_random is not None:
    check('random AP ≈ prevalence', ap_random, yb.mean(), tol=0.03)
else:
    check('random AP ≈ prevalence', None, yb.mean())

<details>
<summary><b>💡 Hint</b></summary>

Sort by score descending; `tp = np.cumsum(y_sorted)`; precision `tp / np.arange(1, n+1)`, recall `tp / y.sum()`; sum the recall increments times precision.

</details>

<details>
<summary><b>✅ Solution</b></summary>

ROC-AUC is about $0.83$ here while AP is only about $0.44$: with 2% positives, even a small false-positive *rate* means many false positives relative to the
few true ones, which ROC hides and precision exposes. The baseline for AP is the **prevalence** (≈0.02 for a random scorer), not 0.5.
(With tied scores, sklearn groups ties into one threshold; with continuous scores the simple version matches exactly.)

```python
X_imb, y_imb = make_classification(n_samples=5000, n_features=10, weights=[0.98], flip_y=0.01, random_state=1)
Xa, Xb_, ya, yb = train_test_split(X_imb, y_imb, test_size=0.5, stratify=y_imb, random_state=0)
scores_imb = LogisticRegression(max_iter=1000).fit(Xa, ya).decision_function(Xb_)

def average_precision(y, scores):
    ys = y[np.argsort(-scores, kind='stable')]
    tp = np.cumsum(ys)
    prec = tp / np.arange(1, len(ys) + 1)
    rec = tp / ys.sum()
    return np.sum(np.diff(np.r_[0, rec]) * prec)

ap_model = average_precision(yb, scores_imb)
ap_random = average_precision(yb, rng.uniform(size=len(yb)))
print('ROC-AUC', roc_auc_score(yb, scores_imb), 'AP', ap_model, 'random AP', ap_random)
```

</details>

### Exercise 10 · Always compare to a baseline
*💻 Code · ★☆☆*

A teammate is proud of a regression model with test MAE $\approx 8.4$. Compute the MAE of the trivial baseline that predicts the **training
median** (`mae_dummy`, use `DummyRegressor`), the model's MAE (`mae_model`), and the skill score $1-\mathrm{MAE}_{\text{model}}/\mathrm{MAE}_{\text{dummy}}$ (`skill`).

In [ ]:
x_b = rng.normal(size=(1000, 1))
y_b = 50 + 1.0 * x_b[:, 0] + 10 * rng.normal(size=1000)
xb_tr, xb_te, yb_tr, yb_te = train_test_split(x_b, y_b, test_size=0.3, random_state=0)
pred_model = LinearRegression().fit(xb_tr, yb_tr).predict(xb_te)
mae_model = None
mae_dummy = None
skill = None

_dm = mean_absolute_error(yb_te, np.full(len(yb_te), np.median(yb_tr)))
check('dummy MAE', mae_dummy, _dm)
check('skill score', skill, 1 - mean_absolute_error(yb_te, pred_model) / _dm)

<details>
<summary><b>💡 Hint</b></summary>

`DummyRegressor(strategy='median').fit(xb_tr, yb_tr).predict(xb_te)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The dummy MAE is also about 8.3, so the skill score is about 0 (here even slightly negative): the model is no better than "always predict
the median", because the target is almost pure noise.
A metric means nothing without a baseline: always report the dummy and a simple model next to your fancy one.

```python
x_b = rng.normal(size=(1000, 1))
y_b = 50 + 1.0 * x_b[:, 0] + 10 * rng.normal(size=1000)
xb_tr, xb_te, yb_tr, yb_te = train_test_split(x_b, y_b, test_size=0.3, random_state=0)
pred_model = LinearRegression().fit(xb_tr, yb_tr).predict(xb_te)
mae_model = mean_absolute_error(yb_te, pred_model)
mae_dummy = mean_absolute_error(yb_te, DummyRegressor(strategy='median').fit(xb_tr, yb_tr).predict(xb_te))
skill = 1 - mae_model / mae_dummy
print(mae_model, mae_dummy, skill)
```

</details>

### Exercise 11 · Misreading t-SNE
*🧠 Concept · ★★☆*

A t-SNE plot of your embeddings shows three clusters of equal size, two of them close together and one far away. A colleague concludes:
(a) the three groups have equal spread, (b) the two nearby groups are more similar to each other than to the third, and (c) since a fourth
run with another seed looks different, the embedding is broken. Assess each claim ([[Dimensionality Reduction]]).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

t-SNE preserves *local neighbourhoods*; it adapts the bandwidth per point (perplexity) and optimises a non-convex objective.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Not supported**: t-SNE equalises density through per-point bandwidths, so a tight and a diffuse cluster can look the same size.
(b) **Not reliable**: distances *between* clusters are not preserved; only membership in local neighbourhoods is. Check in the original space (e.g. centroid distances) or with PCA/UMAP.
(c) **Wrong conclusion**: the objective is non-convex, so different seeds give different layouts (rotations, cluster positions). Compare several seeds and perplexities; stable structure is meaningful, layout details are not. Never use t-SNE coordinates as features for a downstream model.

</details>

### Exercise 12 · Correlation is not causation
*🧠 Concept · ★☆☆*

Your churn model's most important feature is "number of support tickets": more tickets, more churn. The product manager wants to make it
harder to open tickets to reduce churn. What is wrong, and how could you find out whether tickets *cause* churn?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

What could cause both tickets and churn? What does feature importance actually measure?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Feature importance measures **predictive** usefulness, not causal effect. A confounder (e.g. a buggy product experience) likely causes both
tickets and churn; hiding the ticket button removes the signal, not the problem, and may even increase churn. To estimate the causal effect, run a
randomised experiment (A/B test) or use observational methods (adjustment for confounders, IPW, quasi-experiments): see [[Causal Inference]].

</details>

## Part D · Code-level bugs

### Exercise 13 · The broadcasting bug
*💻 Code · ★☆☆*

This MSE is silently wrong. Find out why: store the shape of the intermediate `err` in `buggy_shape`, then write a correct `mse_fixed`.

In [ ]:
y_col = rng.normal(size=(200, 1))       # e.g. loaded from a DataFrame column with [[...]]
y_hat = y_col[:, 0] + 0.1 * rng.normal(size=200)

err = y_col - y_hat                     # the bug
mse_buggy = np.mean(err ** 2)
print('buggy MSE:', mse_buggy)

buggy_shape = None
mse_fixed = None

check('shape of err', buggy_shape, (200, 200))
check('fixed MSE', mse_fixed, mean_squared_error(y_col.ravel(), y_hat))

<details>
<summary><b>💡 Hint</b></summary>

$(200,1) - (200,)$ broadcasts to $(200,200)$: every target minus every prediction. Flatten one side, and `assert err.shape == y_hat.shape`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Broadcasting turns the difference into a $200\times200$ matrix of all pairs, so the "MSE" is about $1.5$ instead of $0.01$, without any error.
Fix: `y_col.ravel()` (or `y_hat[:, None]`), and add shape assertions in every loss function ([[NumPy Snippets]]).

```python
y_col = rng.normal(size=(200, 1))
y_hat = y_col[:, 0] + 0.1 * rng.normal(size=200)
err = y_col - y_hat
mse_buggy = np.mean(err ** 2)
buggy_shape = err.shape
err_ok = y_col.ravel() - y_hat
assert err_ok.shape == y_hat.shape
mse_fixed = np.mean(err_ok ** 2)
print(mse_buggy, mse_fixed)
```

</details>

### Exercise 14 · Train vs. eval mode: dropout
*💻 Code · ★★☆*

Implement **inverted dropout** `dropout(x, p, train, rng)`: in training, zero each unit with probability $p$ and scale survivors by
$1/(1-p)$; in evaluation, return $x$ unchanged. Verify that the expectation is preserved in training and explain what goes wrong if you forget
`model.eval()` at test time, or if you implement non-inverted dropout but forget to rescale at test time.

In [ ]:
def dropout(x, p, train, rng):
    return None

x_act = np.ones((1000, 100))
out_train = dropout(x_act, 0.3, True, np.random.default_rng(1))
out_eval = dropout(x_act, 0.3, False, np.random.default_rng(1))

check('eval mode is the identity', out_eval, x_act)
check('train mode keeps the mean', None if out_train is None else out_train.mean(), 1.0, tol=0.01)
check('train mode drops ~30%', None if out_train is None else np.mean(out_train == 0), 0.3, tol=0.01)

<details>
<summary><b>💡 Hint</b></summary>

`mask = rng.uniform(size=x.shape) >= p`; return `x * mask / (1 - p)` in training.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbb E[\text{out}] = (1-p)\cdot\frac{x}{1-p}=x$, so no rescaling is needed at test time. Forgetting eval mode makes predictions **random**
(different on every call) and worse on average; with BatchNorm, it also uses the test batch's statistics instead of the running averages ([[PyTorch Recipes]]).
With *non-inverted* dropout and no test-time scaling, activations at test time are $1/(1-p)$ times larger than during training.

```python
def dropout(x, p, train, rng):
    if not train:
        return x
    mask = rng.uniform(size=x.shape) >= p
    return x * mask / (1 - p)

x_act = np.ones((1000, 100))
out_train = dropout(x_act, 0.3, True, np.random.default_rng(1))
out_eval = dropout(x_act, 0.3, False, np.random.default_rng(1))
```

</details>

### Exercise 15 · Reproducibility checklist
*🧠 Concept · ★☆☆*

You rerun last week's notebook and get 87.1% instead of 88.4%. List at least five sources of non-reproducibility and what you should log
or fix for each.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Randomness, data, code, environment, hardware.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Random seeds**: NumPy/`random`/framework seeds, `random_state` in splits and models; log them.
2. **Data**: the dataset changed (new rows, different preprocessing); version or hash the data.
3. **Code and config**: hyperparameters, git commit; log a config file per run (MLflow, W&B…).
4. **Environment**: library versions (sklearn defaults change between versions!); pin with a lock file / container.
5. **Hardware/nondeterministic kernels**: GPU atomics, multithreading, `cudnn.benchmark`; enable deterministic algorithms if needed.
6. **Evaluation variance**: a 1.3-point difference may be within the run-to-run noise; report mean ± std over several seeds.

</details>

### Exercise 16 · Log-sum-exp by hand
*✍️ By hand · ★☆☆*

Compute $\mathrm{LSE}(z)=\log\sum_i e^{z_i}$ for $z=(1000, 1000)$ by hand, using $\mathrm{LSE}(z)=m+\log\sum_i e^{z_i-m}$ with $m=\max_i z_i$.
What does `np.log(np.sum(np.exp(z)))` return, and why? Also give $\log\mathrm{softmax}(z)_1$.

In [ ]:
lse_hand = None
log_softmax_1 = None

check('LSE(1000, 1000)', lse_hand, sp_logsumexp([1000., 1000.]))
check('log softmax', log_softmax_1, 1000 - sp_logsumexp([1000., 1000.]))

<details>
<summary><b>💡 Hint</b></summary>

$e^{1000}$ overflows float64 (max ≈ $1.8\cdot10^{308}\approx e^{709.8}$).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathrm{LSE}=1000+\log(e^0+e^0)=1000+\ln2=1000.6931$. The naive version computes $e^{1000}=\infty$ and returns `inf` (with an overflow warning).
$\log\mathrm{softmax}(z)_1 = z_1-\mathrm{LSE}(z) = -\ln 2 = -0.6931$.

```python
lse_hand = 1000 + np.log(2)
log_softmax_1 = -np.log(2)
```

</details>

### Exercise 17 · Numerically stable losses
*💻 Code · ★★☆*

Implement `logsumexp(z)` (1-D, stable) and `bce_with_logits(z, y)`: the mean binary cross-entropy computed **from logits** without ever forming
$\sigma(z)$, using $\ell = \max(z,0) - zy + \log(1+e^{-|z|})$. The starter shows the naive version producing `nan`/`inf` for confident predictions.

In [ ]:
z_big = np.array([1000., 999., -5.])
z_log = np.array([40., -40., 3., -800.])
y_log = np.array([0., 1., 1., 0.])
with np.errstate(over='ignore', divide='ignore', invalid='ignore'):
    p_naive = 1 / (1 + np.exp(-z_log))
    print('naive BCE:', -np.mean(y_log * np.log(p_naive) + (1 - y_log) * np.log(1 - p_naive)))

def logsumexp(z):
    return None

def bce_with_logits(z, y):
    return None

check('stable logsumexp', logsumexp(z_big), sp_logsumexp(z_big))
check('stable BCE from logits', bce_with_logits(z_log, y_log), -np.mean(y_log * log_expit(z_log) + (1 - y_log) * log_expit(-z_log)))

<details>
<summary><b>💡 Hint 1</b></summary>

`m = z.max(); return m + np.log(np.sum(np.exp(z - m)))`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

The BCE formula comes from $-y\log\sigma(z)-(1-y)\log(1-\sigma(z)) = \log(1+e^{z})-zy$ and $\log(1+e^{z})=\max(z,0)+\log(1+e^{-|z|})$; use `np.log1p`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\sigma(40)$ rounds to exactly $1.0$ in float64, so $\log(1-\sigma(40))=-\infty$: the naive loss is `inf`. Working in logit space keeps everything finite
(here $\ell=(40 + 40 + 0.0486 + 0)/4\approx20.01$). This is why frameworks offer `BCEWithLogitsLoss` / `cross_entropy` on logits. When you must use probabilities,
clip them to $[\epsilon, 1-\epsilon]$.

```python
z_big = np.array([1000., 999., -5.])
z_log = np.array([40., -40., 3., -800.])
y_log = np.array([0., 1., 1., 0.])

def logsumexp(z):
    m = np.max(z)
    return m + np.log(np.sum(np.exp(z - m)))

def bce_with_logits(z, y):
    return np.mean(np.maximum(z, 0) - z * y + np.log1p(np.exp(-np.abs(z))))

print(logsumexp(z_big), bce_with_logits(z_log, y_log))
```

</details>

## Part E · Capstone

### Exercise 18 · Debug the oversampling pipeline
*💻 Code · ★★★*

Pure-noise features, ~10% positives. The starter **oversamples the minority class before cross-validation** (duplicating positives until balanced)
and reports a great balanced accuracy for 1-NN. Explain the bug, then write the correct protocol in `fixed_bal_acc`: inside each
`StratifiedKFold(5, shuffle=True, random_state=0)` fold of the **original** data, oversample only the training part, fit, and score on the untouched
validation part.

In [ ]:
n_os = 300
X_os = rng.normal(size=(n_os, 20))
y_os = (rng.uniform(size=n_os) < 0.1).astype(int)

def oversample(X, y, r):
    pos = np.flatnonzero(y == 1)
    extra = r.choice(pos, (y == 0).sum() - len(pos))
    return np.r_[X, X[extra]], np.r_[y, y[extra]]

# --- the bug
Xb_os, yb_os = oversample(X_os, y_os, np.random.default_rng(0))
_sc = [balanced_accuracy_score(yb_os[te], KNeighborsClassifier(1).fit(Xb_os[tr], yb_os[tr]).predict(Xb_os[te]))
       for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(Xb_os, yb_os)]
leaky_bal_acc = np.mean(_sc)
print('leaky balanced accuracy:', leaky_bal_acc)

fixed_bal_acc = None

check('leaky protocol is inflated', leaky_bal_acc > 0.75, True)
if fixed_bal_acc is not None:
    check('fixed protocol is at chance', abs(fixed_bal_acc - 0.5) < 0.12, True)
else:
    check('fixed protocol is at chance', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Duplicated positives land in both training and validation folds; what does 1-NN do with an exact copy?

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Loop `for tr, te in skf.split(X_os, y_os)`, call `oversample(X_os[tr], y_os[tr], r)`, fit, and score on `X_os[te]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Copies of the same positive end up on both sides of a split, so 1-NN finds its exact duplicate at distance 0 and "predicts" the minority class
perfectly: balanced accuracy ≈ 0.9 on noise. This is leakage through duplicates (the same as Exercise 5). Any resampling (oversampling, SMOTE,
undersampling) must happen **inside** the training folds, e.g. with an `imblearn` `Pipeline`. Done correctly, the score falls to ≈ 0.5, the honest answer.

```python
n_os = 300
X_os = rng.normal(size=(n_os, 20))
y_os = (rng.uniform(size=n_os) < 0.1).astype(int)

def oversample(X, y, r):
    pos = np.flatnonzero(y == 1)
    extra = r.choice(pos, (y == 0).sum() - len(pos))
    return np.r_[X, X[extra]], np.r_[y, y[extra]]

Xb_os, yb_os = oversample(X_os, y_os, np.random.default_rng(0))
_sc = [balanced_accuracy_score(yb_os[te], KNeighborsClassifier(1).fit(Xb_os[tr], yb_os[tr]).predict(Xb_os[te]))
       for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(Xb_os, yb_os)]
leaky_bal_acc = np.mean(_sc)

r = np.random.default_rng(0)
scores = []
for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(X_os, y_os):
    Xt, yt = oversample(X_os[tr], y_os[tr], r)
    scores.append(balanced_accuracy_score(y_os[te], KNeighborsClassifier(1).fit(Xt, yt).predict(X_os[te])))
fixed_bal_acc = np.mean(scores)
print(leaky_bal_acc, fixed_bal_acc)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Common Pitfalls](Common%20Pitfalls.md).*